# '02-A.-Validaciones Contables Marine' - PROGRAM 03

##  Overview
This program is an additional validation over the final database.
The processed database, and the consolidated file of BDX are loaded.
First, the information is groupped by policie and LoB and compare the info in processed database vs BDX.
Second, a validation by claim number is made and compare the info in processed database vs BDX.
Finally, export this validation in a Excel file.

##  Execution Flow
1. Library Import.
2. Path Definition and Macrovariables. 
3. Data import(Processed Database & Consolidated BDX). 
4. Data Process.
5. Final Export.

##  Output
- Excel file with OSLR and payments validation. ({AñoMes}_Validacion_GR_DED.xlsx)

## 1. Library Import

In [1]:
# =============================================================================
# Section 1: Library import
# =============================================================================

# === Optional: Clean all the enviroment prior running === #
%reset -f                                                  
# ======================================================== #
import os
import pandas as pd
#import dtale
import numpy as np
import timeit
from timeit import timeit
import timeit
import sqlite3

start_time = timeit.default_timer() # Timer

## 2. Path definition and macrovariables

In [2]:
# =============================================================================
# Section 2: Path Definition and Macrovariables. 
# =============================================================================

print('===============================================================================================')
print('Inicio del proceso')
start_time = timeit.default_timer() # Timer
# ========== Edit variables month ========== #
AñoMes = 202510
AñoMes_ant = 202509

# ========== Routes definition ========== #

print('===============================================================================================')
#Define your project directory path as a variable
directorio_proyecto =  "C:/Users/KOT14/Documents/Integral/Marine"   #C:/Users/KOT23/Documents/Integral

#Change the current working directory to the project directory
os.chdir(directorio_proyecto) 

#Verify that the current directory is the project directory
print(f"Directorio actual de trabajo: {os.getcwd()}")

#Processed files path
ruta_procesados = rf"{directorio_proyecto}/Procesados/{AñoMes}" 
print(f"Ruta de archivos procesados: {ruta_procesados}")

#Validation files path
route_validations = rf"{directorio_proyecto}/Validaciones/{AñoMes}" 
print(f"Ruta de Validaciones: {route_validations}")



Inicio del proceso
Directorio actual de trabajo: C:\Users\KOT14\Documents\Integral\Marine
Ruta de archivos procesados: C:/Users/KOT14/Documents/Integral/Marine/Procesados/202510
Ruta de Validaciones: C:/Users/KOT14/Documents/Integral/Marine/Validaciones/202510


## 3. Data import

In [3]:
# =============================================================================
# Section 3: Data import
# =============================================================================

# ======== Import of Final Database ======== #
df_final_marine = pd.read_pickle(f'{ruta_procesados}/final/{AñoMes}_Siniestros_Marine.pkl')

# Import of bdx database
# Uncomment if neccesary
#df_bdx_consolidated = pd.read_excel(fr'{ruta_procesados}/consolidado_bdx.xlsx')
#df_bdx_consolidated.to_pickle(fr'{ruta_procesados}/consolidado_bdx.pkl')

df_bdx_consolidated = pd.read_pickle(fr'{ruta_procesados}/consolidado_bdx.pkl')

FileNotFoundError: [Errno 2] No such file or directory: 'C:/Users/KOT14/Documents/Integral/Marine/Procesados/202510/final/202510_Siniestros_Marine.pkl'

## 4. Data process

In [ ]:
# =============================================================================
# Section 4: Data Process
# =============================================================================

# Init the list of legacy bdx(meaning we don´t have new deliveries of information)
list_legacy = ['BJ200001','BJ2000120000','BJ2000120100','M9000325','M9000324','CJ200025','BJ200008','CJ2000250100', 'CJ2000250200','147736177','38417374','13637426',
               '25200 30002933','25200 30006350','25300 30011610'] # Not in legacy but we dont have track of these policies 


# ======= Final Marine Database ======= #

# String convert 
df_final_marine['INWARD POLICY N°'] = df_final_marine['INWARD POLICY N°'].astype(str).str.strip()
# Drop legacy policies from final database
df_final_marine = df_final_marine.loc[~df_final_marine['INWARD POLICY N°'].isin(list_legacy)].reset_index(drop = True) 
# Format Claim Number
df_final_marine['CLAIM NUMBER'] = df_final_marine['CLAIM NUMBER'].apply(lambda x: str(x).strip().replace(' ', '') if not pd.isna(x) else x)


# ======= BDX Database ======= #

# Drop legacy policies from bdx database
df_bdx_consolidated = df_bdx_consolidated.loc[~df_bdx_consolidated['INWARD POLICY N°'].isin(list_legacy)].reset_index(drop = True) 
# Normalization of name
df_bdx_consolidated.loc[df_bdx_consolidated['INWARD POLICY N°']== '90600 00320575','INWARD POLICY N°'] = '90600 320575'
# Temp: Until we have complete info
df_bdx_consolidated.loc[df_bdx_consolidated['CLAIM NUMBER']== '1980/2024', 'LoB-Inward'] = 'RC FLETADORES(PMI)'
# Fill NaN values with 0
df_bdx_consolidated[['GROSS RESERVE', 'DEDUCTIBLE']] = df_bdx_consolidated[['GROSS RESERVE', 'DEDUCTIBLE']].fillna(0) 
# Format claim number
df_bdx_consolidated['CLAIM NUMBER'] = df_bdx_consolidated['CLAIM NUMBER'].apply(lambda x: str(x).strip().replace(' ', '') if not pd.isna(x) else x)

### 1 Validation

In [ ]:
# =============================================================================
# Section 4: Data Process
# =============================================================================


# ======= First Validation ======= #

# ===== Final Database Processing ===== #

df_totales_marine = df_final_marine.groupby(['INWARD POLICY N°','LoB-Inward']).agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum'
}).reset_index()

#Initiliaze key merge
df_totales_marine['KEY'] = df_totales_marine['INWARD POLICY N°'] + "-" + df_totales_marine['LoB-Inward'] 


# ===== BDX Processing ===== #
df_totales_bdx = df_bdx_consolidated.groupby(['INWARD POLICY N°','LoB-Inward']).agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum'
}).reset_index()

#Initiliaze key merge
df_totales_bdx['KEY'] = df_totales_bdx['INWARD POLICY N°'] + "-" + df_totales_bdx['LoB-Inward']
# Flag to identify the cross
df_totales_bdx['FLAG'] = 1


# ===== Final Merge ===== #
print("=" * 90)
print(f"Tenemos {df_totales_marine.shape[0]} registros en la base de totales Marine antes del merge")

# Perform the merge
df_totales_merged = df_totales_marine.merge(
    df_totales_bdx,
    on = 'KEY',
    how = 'left',
    suffixes = ('', ' BDX')
).reset_index(drop = True)

print(f"Tenemos {df_totales_merged.shape[0]} registros en la base de totales Marine despues del merge")
print((f"Tenemos que cruzaron {df_totales_merged['FLAG'].count()} registros" ))

# ===== Final Format ===== #

# Drop of columns
df_totales_merged.drop(columns = ['KEY','INWARD POLICY N° BDX','LoB-Inward BDX'], inplace = True)

df_totales_merged['VALIDACION GR'] = df_totales_merged['GROSS RESERVE BDX'] - df_totales_merged['GROSS RESERVE']
df_totales_merged['VALIDACION DED'] = df_totales_merged['DEDUCTIBLE BDX'] - df_totales_merged['DEDUCTIBLE']


#df_totales_merged.to_excel(f'{ruta_procesados}/pruebas/{AñoMes}_Validacion_GR_DED.xlsx', index= False)



Tenemos 31 registros en la base de totales Marine antes del merge
Tenemos 31 registros en la base de totales Marine despues del merge
Tenemos que cruzaron 20 registros


### 2 Validation

In [ ]:
# =============================================================================
# Section 4: Data Process
# =============================================================================


# ======= Second Validation ======= #

# ===== Final Database Processing ===== #

df_totales_marine_2 = df_final_marine.groupby(['INWARD POLICY N°','LoB-Inward','CLAIM NUMBER']).agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum'
}).reset_index()

# Initiliaze key merge
df_totales_marine_2['KEY'] = df_totales_marine_2['INWARD POLICY N°'] + "-" + df_totales_marine_2['LoB-Inward'] + "-" + df_totales_marine_2['CLAIM NUMBER'] 


# ===== BDX Processing ===== #
df_totales_bdx_2 = df_bdx_consolidated.groupby(['INWARD POLICY N°','LoB-Inward','CLAIM NUMBER']).agg({
    'GROSS RESERVE': 'sum',
    'DEDUCTIBLE': 'sum'
}).reset_index()

# Initiliaze key merge
df_totales_bdx_2['KEY'] = df_totales_bdx_2['INWARD POLICY N°'] + "-" + df_totales_bdx_2['LoB-Inward'] + "-" + df_totales_bdx_2['CLAIM NUMBER']
# Flag to identify the cross
df_totales_bdx_2['FLAG'] = 1


# ===== Final Merge ===== #
print("=" * 90)
print(f"Tenemos {df_totales_marine_2.shape[0]} registros en la base de totales Marine antes del merge")

# Perform the merge
df_totales_merged_2 = df_totales_marine_2.merge(
    df_totales_bdx_2,
    on = 'KEY',
    how = 'left',
    suffixes = ('', ' BDX')
).reset_index(drop = True)

print(f"Tenemos {df_totales_merged_2.shape[0]} registros en la base de totales Marine despues del merge")
print((f"Tenemos que cruzaron {df_totales_merged_2['FLAG'].count()} registros" ))


# ===== Final Format ===== #
df_totales_merged_2.drop(columns = ['KEY' ,'LoB-Inward BDX', 'CLAIM NUMBER BDX'], inplace = True)

df_totales_merged_2['VALIDACION GR'] = df_totales_merged_2['GROSS RESERVE BDX'] - df_totales_merged_2['GROSS RESERVE']
df_totales_merged_2['VALIDACION DED'] = df_totales_merged_2['DEDUCTIBLE BDX'] - df_totales_merged_2['DEDUCTIBLE']


Tenemos 2052 registros en la base de totales Marine antes del merge
Tenemos 2052 registros en la base de totales Marine despues del merge
Tenemos que cruzaron 1557 registros


## 5. Final Export

In [ ]:
# =============================================================================
# Section 5: Final Export 
# =============================================================================
output_final = f'{route_validations}/{AñoMes}_Validacion_GR_DED.xlsx'

#ExcelWriter for final output
with pd.ExcelWriter(output_final, engine= 'xlsxwriter') as writer:
    # Export the first validation to 'TOTALES'
    df_totales_merged.to_excel(writer, sheet_name = 'TOTALES', index = False)
    # Export the second validation to 'DETALLE'
    df_totales_merged_2.to_excel(writer, sheet_name = 'DETALLE', index = False)

print(f'Se exportó el archivo de validacion de Gross Reserve y Deducible al año {AñoMes}')


Se exportó el archivo de validacion de Gross Reserve y Deducible al año 202509


In [ ]:
end_time = timeit.default_timer()
print(f"El programa terminó de correr correctamente en: {round(end_time - start_time,2)} s")

El programa terminó de correr correctamente en: 0.47 s
